# Replicating section 3 of *Parallel and Distributed Deep Learning* (Hegde & Usmani) on Kaggle

CPU vs GPU forward time of the paper's conv layer (Fig 1–2) and of the FC layer (Fig 3–4),
measured **on the same machine**, like the paper did on one AWS instance.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (recent PyTorch builds may no longer support the P100).
2. Run All (or Save Version → Save & Run All). Three full runs, roughly 35 minutes in total.
3. Download `results_kaggle.zip` from the Output panel, then stop the session (power button).

Any failing step raises an error, so Run All stops there instead of carrying on.

The code cells below are generated from the project's `code/` folder by `make_kaggle_notebook.py`.

## 1 · Machine check

In [ ]:
!nvidia-smi
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core|Socket'
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2, then restart the session."
print("GPU 0:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
print("torch CPU threads (max):", torch.get_num_threads())
!mkdir -p /kaggle/working/code

## 2 · Benchmark code (written to `/kaggle/working/code`)

In [ ]:
%%writefile /kaggle/working/code/common.py
"""Shared helpers for the benchmarks: devices, threads, timing, machine info, CSV output."""
import csv
import json
import os
import platform
import random
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parent.parent

# torch picks one intra-op thread per physical core by default. Capture it before
# any benchmark calls torch.set_num_threads(), so "max" always means this value.
DEFAULT_THREADS = torch.get_num_threads()

CSV_FIELDS = [
    "exp", "impl", "device", "threads", "size",
    "mean_ms", "std_ms", "min_ms", "median_ms", "issue_ms", "warmup", "repeats",
    "gflops", "gbps", "order", "t_start", "t_end", "machine", "timestamp",
]


def parse_ints(text):
    return [int(v) for v in text.split(",") if v.strip()]


def resolve_threads(spec):
    """'1,max' -> [1, DEFAULT_THREADS] (duplicates removed, order kept)."""
    values = []
    for item in spec.split(","):
        item = item.strip()
        if item == "max":
            values.append(DEFAULT_THREADS)
        elif item:
            values.append(int(item))
    return list(dict.fromkeys(values))


def get_device(name):
    if name == "cuda":
        if not torch.cuda.is_available():
            raise SystemExit("CUDA is not available on this machine.")
        # The paper's GPU (Kepler) only had plain FP32. Turn off TF32 shortcuts so
        # newer GPUs compute in true FP32 too.
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = True
    elif name != "cpu":
        raise SystemExit(f"Unknown device '{name}' (use cpu or cuda).")
    return torch.device(name)


def available_devices():
    return ["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def bench(fn, device, warmup=3, repeat=10):
    """Run fn `warmup` times untimed, then `repeat` timed runs (wall clock, ms).

    GPU work is asynchronous, so the device is synchronized before starting and
    after finishing each timed run: the time covers the full kernel execution.

    issue_ms is the part until fn() returns, i.e. before the final synchronize. On a GPU
    that is the host's launch work (as long as the launches fit in the CUDA queue), so
    issue_ms ~ mean_ms means the run is bound by the CPU launching kernels, and
    issue_ms << mean_ms means it is bound by the GPU executing them.
    """
    for _ in range(warmup):
        fn()
    sync(device)
    times, issue = [], []
    t_start = time.time()
    for _ in range(repeat):
        sync(device)
        start = time.perf_counter()
        fn()
        issue.append((time.perf_counter() - start) * 1000.0)
        sync(device)
        times.append((time.perf_counter() - start) * 1000.0)
    return {
        "issue_ms": statistics.fmean(issue),
        "t_start": t_start,
        "t_end": time.time(),
        "mean_ms": statistics.fmean(times),
        "std_ms": statistics.stdev(times) if len(times) > 1 else 0.0,
        "min_ms": min(times),
        "median_ms": statistics.median(times),
    }


def ordered(items, seed):
    """Measurement order. Seed 0 keeps it ascending; any other seed shuffles reproducibly.

    With an ascending order "larger batch" and "later in time" cannot be told apart, e.g.
    a GPU raising its clocks during the sweep would look like a batch-size effect.
    """
    items = list(items)
    if seed:
        random.Random(seed).shuffle(items)
    return items


def gpu_warmup(device, seconds):
    """Keep the GPU busy so it reaches its sustained clocks before anything is timed."""
    if device.type != "cuda" or seconds <= 0:
        return
    a = torch.randn(4096, 4096, device=device)
    b = torch.empty_like(a)
    end = time.perf_counter() + seconds
    while time.perf_counter() < end:
        torch.mm(a, a, out=b)
        torch.cuda.synchronize(device)
    del a, b


class GpuClockLog:
    """Log GPU 0 clocks/power/temperature (gpu_sampler.py, every 20 ms) while the block runs."""

    def __init__(self, path, enabled, log=print):
        self.path, self.enabled, self.log, self.proc = Path(path), enabled, log, None

    def __enter__(self):
        if self.enabled:
            self.err = self.path.with_suffix(".err")
            self.err_fh = self.err.open("w", encoding="utf-8")
            self.proc = subprocess.Popen(
                [sys.executable, str(Path(__file__).with_name("gpu_sampler.py")), str(self.path), "20"],
                stderr=self.err_fh)
            time.sleep(1)
        return self

    def __exit__(self, *exc):
        if self.proc is None:
            return
        self.proc.terminate()
        self.proc.wait(timeout=10)
        self.err_fh.close()
        samples = max(len(self.path.read_text().splitlines()) - 1, 0) if self.path.exists() else 0
        error = self.err.read_text(encoding="utf-8").strip().splitlines()
        self.log(f"gpu clock log: {samples} samples" + (f" | ERROR: {error[-1]}" if error else ""))


_order = 0


def make_row(exp, impl, device, threads, size, stats, warmup, repeat, flops, nbytes, machine):
    global _order
    _order += 1
    seconds = stats["mean_ms"] / 1000.0
    return {
        "exp": exp,
        "impl": impl,
        "device": device.type,
        "threads": threads,
        "size": size,
        **{k: round(v, 6) for k, v in stats.items()},
        "warmup": warmup,
        "repeats": repeat,
        "gflops": round(flops / seconds / 1e9, 3) if flops else "",
        "gbps": round(nbytes / seconds / 1e9, 3) if nbytes else "",
        "order": _order,
        "machine": machine,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def append_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new_file = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerows(rows)


def cpu_name():
    system = platform.system()
    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                 r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()
        except OSError:
            pass
    elif system == "Linux":
        try:
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
        except OSError:
            pass
    return platform.processor() or "unknown"


def env_info(machine):
    info = {
        "machine": machine,
        "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "os": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu": cpu_name(),
        "cpu_logical_cores": os.cpu_count(),
        "torch_default_threads": DEFAULT_THREADS,
        "mkl": torch.backends.mkl.is_available(),
        "mkldnn": torch.backends.mkldnn.is_available(),
        "cuda_available": torch.cuda.is_available(),
        "malloc_env": {k: v for k, v in os.environ.items() if k.startswith("MALLOC_")},
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update({
            "gpu": props.name,
            "gpu_count": torch.cuda.device_count(),
            "gpu_memory_gb": round(props.total_memory / 1024**3, 1),
            "gpu_sm_count": props.multi_processor_count,
            "gpu_compute_capability": f"{props.major}.{props.minor}",
            "cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
        })
    return info


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
%%writefile /kaggle/working/code/gpu_sampler.py
"""Sample GPU 0 through NVML into a CSV until killed. Started by common.GpuClockLog.

    python gpu_sampler.py out.csv 20        # every 20 ms

A separate process, so it never competes with the benchmark for the Python GIL. Uses
ctypes on libnvidia-ml (installed with the driver), no extra package needed.
"""
import ctypes
import sys
import time


def check(code, what):
    if code != 0:
        raise RuntimeError(f"NVML {what} failed with code {code}")


def main():
    path, interval = sys.argv[1], float(sys.argv[2]) / 1000
    nvml = ctypes.CDLL("nvml.dll" if sys.platform == "win32" else "libnvidia-ml.so.1")
    check(nvml.nvmlInit_v2(), "init")
    handle = ctypes.c_void_p()
    check(nvml.nvmlDeviceGetHandleByIndex_v2(0, ctypes.byref(handle)), "handle")
    sm, mem, power, temp = (ctypes.c_uint() for _ in range(4))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("t,sm_mhz,mem_mhz,power_w,temp_c\n")
        while True:
            check(nvml.nvmlDeviceGetClockInfo(handle, 1, ctypes.byref(sm)), "SM clock")
            check(nvml.nvmlDeviceGetClockInfo(handle, 2, ctypes.byref(mem)), "memory clock")
            # Power and temperature are extra information: 0 if the driver refuses them.
            nvml.nvmlDeviceGetPowerUsage(handle, ctypes.byref(power))
            nvml.nvmlDeviceGetTemperature(handle, 0, ctypes.byref(temp))
            fh.write(f"{time.time():.4f},{sm.value},{mem.value},{power.value / 1000:.1f},{temp.value}\n")
            fh.flush()
            time.sleep(interval)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/bench_conv.py
"""Experiment 1 (paper Fig 1-2): forward time of the paper's first conv layer vs batch size.

Layer from the paper: greyscale 224x224 input, 96 filters of 11x11, stride 4.
Output is 96 x 54 x 54, so each image unrolls (im2col) into a 121 x 2916 matrix.
(The paper says "54 parts"; it is 54 per side, i.e. 54*54 = 2916 patches.)

Implementations:
  caffe   - what the paper describes for Caffe: loop over the images of the batch one
            at a time; per image im2col (F.unfold) then one GEMM W(96x121) @ cols(121x2916)
            plus bias. This is the main series, comparable to the paper.
  batched - im2col for the whole batch, one batched GEMM (extension).
  conv2d  - the library convolution (oneDNN on CPU, cuDNN on GPU) (extension).

caffe and batched write into preallocated buffers (Caffe keeps its column buffer and output
blob too). F.conv2d has no out= argument, so on CPU it still allocates its output on every
call and the first-touch page faults of that buffer are part of its time. On Linux, run with
MALLOC_MMAP_MAX_=0 and a large MALLOC_TRIM_THRESHOLD_ so malloc reuses memory instead.
"""
import argparse

import torch
import torch.nn.functional as F

from common import (DEFAULT_THREADS, append_rows, bench, get_device, gpu_warmup, make_row,
                    ordered, parse_ints, resolve_threads)

IMG = 224
C_IN = 1
C_OUT = 96
K = 11
STRIDE = 4
OUT_HW = (IMG - K) // STRIDE + 1           # 54
PATCHES = OUT_HW * OUT_HW                   # 2916
PATCH_LEN = C_IN * K * K                    # 121
FLOPS_PER_IMAGE = 2 * C_OUT * PATCH_LEN * PATCHES   # multiply-adds of the GEMM

IMPLS = ("caffe", "batched", "conv2d")
DEFAULT_BATCHES = "1,8,16,32,64,96,128,160,192,224,256"


def make_layer(device):
    gen = torch.Generator().manual_seed(0)
    weight = torch.randn(C_OUT, C_IN, K, K, generator=gen).to(device)
    bias = torch.randn(C_OUT, generator=gen).to(device)
    return weight, bias


def im2col(x, out):
    return torch._C._nn.im2col(x, (K, K), (1, 1), (0, 0), (STRIDE, STRIDE), out=out)


def make_fn(impl, x, weight, bias):
    """Return a zero-argument callable that runs one forward pass of the layer."""
    w2 = weight.view(C_OUT, PATCH_LEN)        # unrolled kernels, 96 x 121
    b2 = bias.view(C_OUT, 1)
    n = x.shape[0]

    if impl == "caffe":
        out = torch.empty(n, C_OUT, PATCHES, device=x.device)       # Caffe's top blob
        cols = torch.empty(1, PATCH_LEN, PATCHES, device=x.device)  # Caffe's column buffer

        def fn():
            for i in range(n):
                im2col(x[i:i + 1], cols)                            # 121 x 2916
                torch.addmm(b2, w2, cols[0], out=out[i])            # 96 x 2916
            return out
    elif impl == "batched":
        out = torch.empty(n, C_OUT, PATCHES, device=x.device)
        cols = torch.empty(n, PATCH_LEN, PATCHES, device=x.device)

        def fn():
            im2col(x, cols)                                         # n x 121 x 2916
            return torch.matmul(w2, cols, out=out).add_(b2)         # n x 96 x 2916
    elif impl == "conv2d":
        def fn():
            return F.conv2d(x, weight, bias, stride=STRIDE)
    else:
        raise ValueError(f"unknown impl {impl}")
    return fn


def check_correctness(device):
    """All three implementations must give the same output before anything is timed."""
    weight, bias = make_layer(device)
    x = torch.randn(3, C_IN, IMG, IMG, generator=torch.Generator().manual_seed(1)).to(device)
    with torch.inference_mode():
        ref = make_fn("conv2d", x, weight, bias)().reshape(3, C_OUT, PATCHES)
        for impl in ("caffe", "batched"):
            got = make_fn(impl, x, weight, bias)().reshape(3, C_OUT, PATCHES)
            torch.testing.assert_close(got, ref, rtol=1e-3, atol=1e-2,
                                       msg=lambda m, impl=impl: f"{impl} != conv2d on {device}: {m}")


def run(device_name, threads_list, impls, batches, warmup, repeat, machine, out_csv, log=print,
        order_seed=0, warm_seconds=0):
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]      # host threads do not matter for GPU runs
    check_correctness(device)
    weight, bias = make_layer(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        for impl in impls:
            gpu_warmup(device, warm_seconds)
            for n in ordered(batches, order_seed):
                x = torch.randn(n, C_IN, IMG, IMG, device=device)
                with torch.inference_mode():
                    fn = make_fn(impl, x, weight, bias)
                    stats = bench(fn, device, warmup, repeat)
                flops = FLOPS_PER_IMAGE * n
                nbytes = 4 * n * (IMG * IMG + C_OUT * PATCHES)   # read input + write output
                row = make_row("conv", impl, device, threads, n, stats, warmup, repeat,
                               flops, nbytes, machine)
                rows.append(row)
                log(f"conv  {device.type:4s} t={threads:<2d} {impl:7s} batch={n:<4d} "
                    f"{stats['mean_ms']:10.3f} ms ± {stats['std_ms']:.3f}  ({row['gflops']} GFLOP/s)"
                    + (f"  issue {stats['issue_ms']:.3f} ms" if device.type == "cuda" else ""))
                del x, fn
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return rows


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--device", default="cpu", choices=["cpu", "cuda"])
    parser.add_argument("--threads", default="1,max", help="CPU thread counts, e.g. 1,4,max")
    parser.add_argument("--impls", default=",".join(IMPLS))
    parser.add_argument("--batches", default=DEFAULT_BATCHES)
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repeat", type=int, default=10)
    parser.add_argument("--machine", default="local")
    parser.add_argument("--out", default="conv.csv")
    args = parser.parse_args()
    run(args.device, resolve_threads(args.threads), args.impls.split(","),
        parse_ints(args.batches), args.warmup, args.repeat, args.machine, args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/bench_fc.py
"""Experiment 2 (paper Fig 3-4): fully-connected layer time vs matrix size.

The paper implements the FC layer as a matrix-vector product and reports O(n^2) time.
Its Fig 3 caption says "matrix-matrix multiplication", so both are measured:
  matvec - y = A x, A is n x n (main series, comparable to the paper), O(n^2)
  gemm   - C = A B, all n x n (resolves the caption ambiguity), O(n^3)
"""
import argparse

import torch

from common import (DEFAULT_THREADS, append_rows, bench, get_device, gpu_warmup, make_row,
                    ordered, parse_ints, resolve_threads)

OPS = ("matvec", "gemm")
DEFAULT_SIZES = {
    "matvec": "256,512,1024,2048,4096,8192,12288,16384",
    "gemm": "256,512,1024,1536,2048,3072,4096",
}


def make_fn(op, n, device):
    gen = torch.Generator().manual_seed(n)
    a = torch.randn(n, n, generator=gen).to(device)
    if op == "matvec":
        x = torch.randn(n, generator=gen).to(device)
        y = torch.empty(n, device=device)
        return (lambda: torch.mv(a, x, out=y)), 2 * n * n, 4 * (n * n + 2 * n)
    if op == "gemm":
        b = torch.randn(n, n, generator=gen).to(device)
        c = torch.empty(n, n, device=device)
        return (lambda: torch.mm(a, b, out=c)), 2 * n ** 3, 4 * 3 * n * n
    raise ValueError(f"unknown op {op}")


def check_correctness(device):
    gen = torch.Generator().manual_seed(7)
    a = torch.randn(300, 300, generator=gen)
    x = torch.randn(300, generator=gen)
    b = torch.randn(300, 300, generator=gen)
    with torch.inference_mode():
        torch.testing.assert_close(torch.mv(a.to(device), x.to(device)).cpu(),
                                   (a.double() @ x.double()).float(), rtol=1e-4, atol=1e-3)
        torch.testing.assert_close(torch.mm(a.to(device), b.to(device)).cpu(),
                                   (a.double() @ b.double()).float(), rtol=1e-4, atol=1e-3)


def run(device_name, threads_list, ops, sizes, warmup, repeat, machine, out_csv, log=print,
        order_seed=0, warm_seconds=0):
    """sizes: dict op -> list of n."""
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]
    check_correctness(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        for op in ops:
            gpu_warmup(device, warm_seconds)
            for n in ordered(sizes[op], order_seed):
                fn, flops, nbytes = make_fn(op, n, device)
                with torch.inference_mode():
                    stats = bench(fn, device, warmup, repeat)
                row = make_row("fc", op, device, threads, n, stats, warmup, repeat,
                               flops, nbytes, machine)
                rows.append(row)
                log(f"fc    {device.type:4s} t={threads:<2d} {op:7s} n={n:<6d} "
                    f"{stats['mean_ms']:10.3f} ms ± {stats['std_ms']:.3f}  "
                    f"({row['gflops']} GFLOP/s, {row['gbps']} GB/s)")
                del fn
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return rows


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--device", default="cpu", choices=["cpu", "cuda"])
    parser.add_argument("--threads", default="1,max")
    parser.add_argument("--ops", default=",".join(OPS))
    parser.add_argument("--matvec-sizes", default=DEFAULT_SIZES["matvec"])
    parser.add_argument("--gemm-sizes", default=DEFAULT_SIZES["gemm"])
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repeat", type=int, default=10)
    parser.add_argument("--machine", default="local")
    parser.add_argument("--out", default="fc.csv")
    args = parser.parse_args()
    sizes = {"matvec": parse_ints(args.matvec_sizes), "gemm": parse_ints(args.gemm_sizes)}
    run(args.device, resolve_threads(args.threads), args.ops.split(","), sizes,
        args.warmup, args.repeat, args.machine, args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/run_all.py
"""Run the whole benchmark grid on one machine and save it as one run.

    python run_all.py --machine laptop            # full grid, every available device
    python run_all.py --machine kaggle            # same on Kaggle (CPU + GPU)
    python run_all.py --machine laptop --quick    # smoke test, a few seconds

Output: <outdir>/<machine>/<tag>/{env.json, conv.csv, fc.csv, log.txt}, plus gpu_clocks.csv
when a GPU is measured (NVML, every 20 ms).
outdir defaults to ../results, or /kaggle/working/results on Kaggle (input is read-only).
Then plot with:  python plot.py <that run folder>
"""
import argparse
import time
from datetime import datetime
from pathlib import Path

import bench_conv
import bench_fc
from common import (ROOT, GpuClockLog, available_devices, env_info, parse_ints, resolve_threads,
                    write_json)

QUICK = {
    "batches": "1,4,8",
    "matvec": "256,1024",
    "gemm": "256,512",
    "warmup": 1,
    "repeat": 3,
}


def default_outdir():
    kaggle = Path("/kaggle/working")
    return kaggle / "results" if kaggle.is_dir() else ROOT / "results"


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--machine", required=True, help="label, e.g. laptop or kaggle")
    parser.add_argument("--tag", default=datetime.now().strftime("run_%Y%m%d_%H%M%S"))
    parser.add_argument("--outdir", type=Path, default=default_outdir())
    parser.add_argument("--devices", default=",".join(available_devices()))
    parser.add_argument("--threads", default="1,max", help="CPU thread counts")
    parser.add_argument("--batches", default=bench_conv.DEFAULT_BATCHES)
    parser.add_argument("--matvec-sizes", default=bench_fc.DEFAULT_SIZES["matvec"])
    parser.add_argument("--gemm-sizes", default=bench_fc.DEFAULT_SIZES["gemm"])
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repeat", type=int, default=10)
    parser.add_argument("--skip", default="", help="comma list: conv,fc")
    parser.add_argument("--order-seed", type=int, default=0, help="0 = ascending sizes, else shuffled")
    parser.add_argument("--warm-seconds", type=float, default=5,
                        help="GPU busy time before each implementation/op (0 = cold)")
    parser.add_argument("--quick", action="store_true", help="tiny grid for a smoke test")
    args = parser.parse_args()

    if args.quick:
        args.batches, args.matvec_sizes, args.gemm_sizes = QUICK["batches"], QUICK["matvec"], QUICK["gemm"]
        args.warmup, args.repeat = QUICK["warmup"], QUICK["repeat"]

    run_dir = args.outdir / args.machine / args.tag
    run_dir.mkdir(parents=True, exist_ok=False)
    log_file = (run_dir / "log.txt").open("w", encoding="utf-8")

    def log(msg):
        print(msg, flush=True)
        log_file.write(msg + "\n")
        log_file.flush()

    env = env_info(args.machine)
    env["grid"] = {k: v for k, v in vars(args).items() if k not in ("outdir",)}
    write_json(run_dir / "env.json", env)
    log(f"run folder: {run_dir}")
    log(f"cpu: {env['cpu']} | torch threads (max): {env['torch_default_threads']}"
        + (f" | gpu: {env['gpu']}" if env["cuda_available"] else " | no CUDA GPU"))

    threads = resolve_threads(args.threads)
    skip = set(filter(None, args.skip.split(",")))
    start = time.perf_counter()
    for device in args.devices.split(","):
        # The sampler only runs during GPU work, so it cannot disturb the CPU timings.
        with GpuClockLog(run_dir / "gpu_clocks.csv", device == "cuda", log):
            if "conv" not in skip:
                bench_conv.run(device, threads, list(bench_conv.IMPLS), parse_ints(args.batches),
                               args.warmup, args.repeat, args.machine, run_dir / "conv.csv", log,
                               args.order_seed, args.warm_seconds)
            if "fc" not in skip:
                sizes = {"matvec": parse_ints(args.matvec_sizes), "gemm": parse_ints(args.gemm_sizes)}
                bench_fc.run(device, threads, list(bench_fc.OPS), sizes,
                             args.warmup, args.repeat, args.machine, run_dir / "fc.csv", log,
                             args.order_seed, args.warm_seconds)
    log(f"done in {time.perf_counter() - start:.1f} s -> {run_dir}")
    log_file.close()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/aggregate.py
"""Combine several runs of the same grid into one folder that plot.py understands.

    python aggregate.py <run1> <run2> <run3> --out ../results/kaggle/agg

mean_ms is the mean of the runs' means, run_std_ms the sample std BETWEEN runs. The runs
must come from the same machine and software; with runs from one session this shows
run-to-run noise on one machine, not differences between machines.
"""
import argparse
import json
from pathlib import Path

import pandas as pd

KEY = ["exp", "impl", "device", "threads", "size"]
SAME = ("cpu", "gpu", "torch", "cuda", "python", "torch_default_threads")


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("runs", nargs="+", type=Path)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()

    envs = [json.loads((r / "env.json").read_text(encoding="utf-8")) for r in args.runs]
    for field in SAME:
        values = {str(e.get(field)) for e in envs}
        if len(values) > 1:
            raise SystemExit(f"runs differ in {field}: {sorted(values)} - do not average them")

    args.out.mkdir(parents=True, exist_ok=True)
    for name in ("conv", "fc"):
        if not all((r / f"{name}.csv").exists() for r in args.runs):
            continue
        raw = pd.concat([pd.read_csv(r / f"{name}.csv").assign(run=r.name) for r in args.runs],
                        ignore_index=True)
        counts = raw.groupby(KEY)["run"].nunique()
        if (counts != len(args.runs)).any():
            raise SystemExit(f"{name}: some points are missing in some runs:\n{counts[counts != len(args.runs)]}")
        raw.to_csv(args.out / f"runs_{name}.csv", index=False)
        agg = raw.groupby(KEY, as_index=False).agg(
            mean_ms=("mean_ms", "mean"), run_std_ms=("mean_ms", "std"), std_ms=("std_ms", "mean"),
            issue_ms=("issue_ms", "mean"), gflops=("gflops", "mean"), gbps=("gbps", "mean"),
            repeats=("repeats", "max"), runs=("run", "nunique"), machine=("machine", "first"))
        agg.to_csv(args.out / f"{name}.csv", index=False)
    env = dict(envs[0], aggregated_runs=[r.name for r in args.runs])
    (args.out / "env.json").write_text(json.dumps(env, indent=2, ensure_ascii=False), encoding="utf-8")
    print("wrote", args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/gpu_diag.py
"""Why does the per-image GPU time change during a sweep? One table per run.

    python gpu_diag.py <run> [<run> ...] --out gpu_diag.md

For the Caffe-style conv on the GPU, in measurement order: per-image wall time, per-image
host issue time (launching kernels) and the mean SM clock NVML reported in that interval.
  wall changes together with the SM clock        -> the GPU clock is the cause
  wall ~ issue and both change, clock constant   -> the host (kernel launches) is the cause
"""
import argparse
from pathlib import Path

import pandas as pd


def clocks(run_dir):
    path = run_dir / "gpu_clocks.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path, on_bad_lines="skip").apply(pd.to_numeric, errors="coerce").dropna()
    return None if df.empty else df


def table(run_dir):
    conv = pd.read_csv(run_dir / "conv.csv")
    d = conv[(conv["device"] == "cuda") & (conv["impl"] == "caffe")].sort_values("order")
    log = clocks(run_dir)
    lines = [f"## {run_dir.name}", "",
             "| Thứ tự đo | Batch | Wall (µs/ảnh) | Issue (µs/ảnh) | Issue/Wall | SM clock (MHz) | Số mẫu |",
             "|---|---|---|---|---|---|---|"]
    for _, r in d.iterrows():
        sm, count = "?", 0
        if log is not None:
            hit = log[(log["t"] >= r["t_start"]) & (log["t"] <= r["t_end"])]
            if hit.empty:                       # shorter than the 20 ms sampling: take the nearest
                hit = log.iloc[[(log["t"] - r["t_start"]).abs().argmin()]]
            else:
                count = len(hit)
            sm = f"{hit['sm_mhz'].mean():.0f}"
        wall, issue = r["mean_ms"] / r["size"] * 1000, r["issue_ms"] / r["size"] * 1000
        lines.append(f"| {int(r['order'])} | {int(r['size'])} | {wall:.1f} | {issue:.1f} | "
                     f"{issue / wall:.2f} | {sm} | {count} |")
    return lines + ["", "Số mẫu 0: khoảng đo ngắn hơn 20 ms, lấy mẫu xung nhịp gần nhất.", ""]


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("runs", nargs="+", type=Path)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()
    text = "\n".join(["# Chẩn đoán GPU: conv kiểu Caffe", ""] + [l for r in args.runs for l in table(r)])
    args.out.write_text(text, encoding="utf-8")
    print(text)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/plot.py
"""Plot one benchmark run as the paper's Fig 1-4 (plus extensions) and write a summary.

    python plot.py ../results/laptop/run_20261121_101500
    python plot.py <run folder> --baseline-threads 1 --figdir ../figures/laptop

Figures (PNG, in <run folder>/figures unless --figdir is given):
  fig1_conv_time      conv forward time vs batch size, Caffe-style impl, linear fit   (paper Fig 1)
  fig2_conv_speedup   speedup over the CPU baseline vs batch size                     (paper Fig 2)
  fig3_fc_time        matrix-vector time vs n, log-log, fitted exponent               (paper Fig 3)
  fig4_fc_speedup     matrix-vector speedup over the CPU baseline vs n                (paper Fig 4)
  fig5_conv_impls     time per image of caffe / batched / conv2d, one panel per device (extension)
  fig6_gemm_time      matrix-matrix time vs n, log-log                                (extension)
A folder written by aggregate.py (several runs) gets error bars: +-1 std between runs.
Summary: summary.md (readable) and summary.csv (one row per metric), the table view of the charts.
"""
import argparse
import json
import textwrap
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib import font_manager  # noqa: E402
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator  # noqa: E402

# Paper section 3.2.1: slopes of the CPU and GPU lines in its Fig 1 (its axes are "Time (ms)" vs
# batch size, so ms per image), Xeon E5-2670 vs NVIDIA GRID K520 with Caffe.
PAPER_CPU_SLOPE = 6.45
PAPER_GPU_SLOPE = 0.16

# Validated categorical palette (light surface); color follows the entity:
# slot 1 = CPU baseline, slot 2 = GPU, slot 3+ = other CPU thread counts.
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
IMPL_LABEL = {"caffe": "kiểu Caffe (im2col + GEMM từng ảnh)",
              "batched": "im2col + GEMM cả batch",
              "conv2d": "thư viện (oneDNN / cuDNN)"}
IMPL_ORDER = ["caffe", "batched", "conv2d"]


def apply_style():
    names = {f.name for f in font_manager.fontManager.ttflist}
    font = "Segoe UI" if "Segoe UI" in names else "DejaVu Sans"
    plt.rcParams.update({
        "font.family": font,
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.axisbelow": True,
        "axes.labelcolor": INK2, "axes.labelsize": 10,
        "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 12,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.linestyle": "-",
        "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
        "xtick.labelsize": 9, "ytick.labelsize": 9,
        "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK2,
        "lines.linewidth": 2, "lines.markersize": 6,
    })


# ---------------------------------------------------------------- data

def load_run(run_dir):
    env_path = run_dir / "env.json"
    env = json.loads(env_path.read_text(encoding="utf-8")) if env_path.exists() else {}
    frames = {}
    for name in ("conv", "fc"):
        path = run_dir / f"{name}.csv"
        if path.exists():
            df = pd.read_csv(path)
            df["config"] = np.where(df["device"] == "cuda", "cuda", "cpu-" + df["threads"].astype(str))
            frames[name] = df
        raw = run_dir / f"runs_{name}.csv"          # only in a folder written by aggregate.py
        if raw.exists():
            df = pd.read_csv(raw)
            df["config"] = np.where(df["device"] == "cuda", "cuda", "cpu-" + df["threads"].astype(str))
            frames[f"runs_{name}"] = df
    return env, frames


def series_meta(keys, env, baseline):
    """key -> (label, color, sort order). Fixed slots so colors match across machines."""
    gpu = env.get("gpu", "GPU")
    meta = {}
    others = sorted((k for k in keys if k.startswith("cpu-") and k != baseline), key=lambda k: int(k[4:]))
    if baseline in keys:
        meta[baseline] = (f"CPU · {baseline[4:]} luồng", SLOTS[0], 0)
    if "cuda" in keys:
        meta["cuda"] = (f"GPU · {gpu}", SLOTS[1], 1)
    for i, key in enumerate(others):
        meta[key] = (f"CPU · {key[4:]} luồng", SLOTS[min(2 + i, len(SLOTS) - 1)], 2 + i)
    return dict(sorted(meta.items(), key=lambda kv: kv[1][2]))


def linfit(x, y):
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - float(np.sum((y - pred) ** 2)) / ss_tot if ss_tot > 0 else 1.0
    return float(slope), float(intercept), r2


def powfit(n, t, min_n=1024):
    """Exponent k of t ~ c * n^k, fitted on the larger sizes (small ones are overhead-bound)."""
    mask = n >= min_n
    if mask.sum() < 3:
        mask = np.ones_like(n, dtype=bool)
    k, _ = np.polyfit(np.log(n[mask]), np.log(t[mask]), 1)
    return float(k)


def speedup_table(df, baseline):
    """Rows: size; columns: config; value = baseline time / config time."""
    pivot = df.pivot_table(index="size", columns="config", values="mean_ms")
    if baseline not in pivot:
        return None
    return pivot.rdiv(pivot[baseline], axis=0).drop(columns=baseline)


# ---------------------------------------------------------------- drawing helpers

def end_labels(ax, items, min_gap=0.07):
    """Direct labels right of the plot: items = [(x, y, text, color)].

    Text stays in ink; a thin leader in the series color ties it to the line end.
    Labels are pushed apart vertically so they never overlap.
    """
    if not items:
        return
    ax.get_xlim(), ax.get_ylim()
    to_axes = ax.transAxes.inverted()
    placed = []
    for x, y, text, color in items:
        frac = to_axes.transform(ax.transData.transform((x, y)))[1]
        placed.append([frac, x, y, text, color])
    placed.sort(key=lambda p: p[0])
    for i in range(1, len(placed)):
        placed[i][0] = max(placed[i][0], placed[i - 1][0] + min_gap)
    if placed[-1][0] > 0.98:                  # pushed above the plot: slide the stack back down
        placed[-1][0] = 0.98
        for i in range(len(placed) - 2, -1, -1):
            placed[i][0] = min(placed[i][0], placed[i + 1][0] - min_gap)
    for frac, x, y, text, color in placed:
        ax.annotate(text, xy=(x, y), xycoords="data", xytext=(1.03, frac), textcoords="axes fraction",
                    va="center", ha="left", fontsize=9, color=INK2, annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color=color, lw=1, shrinkA=0, shrinkB=4))


def top_legend(ax):
    """One-row legend between the title and the plot, so it never covers data."""
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=4, borderaxespad=0.3,
              handletextpad=0.3, columnspacing=1.6)


def reps(df):
    text = f"trung bình {int(df['repeats'].max())} lần đo"
    if "runs" in df and df["runs"].max() > 1:
        text += f" × {int(df['runs'].max())} lần chạy (thanh sai số: ±1 độ lệch chuẩn giữa các lần chạy)"
    return text


def yerr(d):
    return d["run_std_ms"].to_numpy(float) if "run_std_ms" in d else None


def footnote(fig, text, width=135):
    fig.text(0.0, -0.02, textwrap.fill(text, width), ha="left", va="top", fontsize=8, color=MUTED)


def int_ticks(ax, values, log=False):
    """Ticks exactly at the measured sizes, as plain integers (no thousands separator)."""
    if log:
        ax.set_xscale("log", base=2)
    ax.set_xticks(values)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v)}"))
    ax.xaxis.set_minor_locator(NullLocator())


def pow2(values):
    """Only the power-of-two sizes get a tick label (12288 next to 16384 would collide)."""
    ticks = sorted(int(v) for v in values if int(v) > 0 and int(v) & (int(v) - 1) == 0)
    return ticks or sorted(int(v) for v in values)


def log_y(ax):
    """Log y axis with readable plain-number ticks at 1-2-5 steps."""
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_locator(NullLocator())


def save(fig, figdir, name):
    path = figdir / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.25)
    plt.close(fig)
    print("saved", path)


def scatter_line(ax, x, y, color, label, line=True, err=None):
    if err is not None:
        ax.errorbar(x, y, yerr=err, fmt="none", ecolor=color, elinewidth=1, capsize=3, zorder=2.5)
    if line:
        ax.plot(x, y, color=color, lw=2, zorder=2)
    ax.plot(x, y, "o", color=color, ms=6, mec=SURFACE, mew=1.5, label=label, zorder=3)


# ---------------------------------------------------------------- figures

def fig_conv_time(conv, meta, figdir, summary):
    df = conv[conv["impl"] == "caffe"]
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    labels = []
    for key, (label, color, _) in meta.items():
        d = df[df["config"] == key].sort_values("size")
        if d.empty:
            continue
        x, y = d["size"].to_numpy(float), d["mean_ms"].to_numpy(float)
        slope, intercept, r2 = linfit(x, y)
        summary.append(("conv_caffe", key, "slope_ms_per_image", slope))
        summary.append(("conv_caffe", key, "intercept_ms", intercept))
        summary.append(("conv_caffe", key, "r2", r2))
        xs = np.array([0, x.max()])
        ax.plot(xs, slope * xs + intercept, color=color, lw=2, zorder=2)
        scatter_line(ax, x, y, color, label, line=False, err=yerr(d))
        labels.append((x.max(), slope * x.max() + intercept, f"{label}\n{slope:.3g} ms/ảnh · R² {r2:.3f}", color))
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Batch size (số ảnh)")
    ax.set_ylabel("Thời gian forward (ms)")
    ax.set_title("Hình 1 · Lớp convolution: thời gian forward theo batch size", pad=30)
    top_legend(ax)
    end_labels(ax, labels, min_gap=0.12)
    footnote(fig, "Lớp conv của paper: ảnh xám 224×224, 96 filter 11×11, stride 4. Cài đặt kiểu Caffe "
                  f"(im2col + GEMM từng ảnh), {reps(df)}. Đường thẳng: hồi quy tuyến tính, "
                  f"độ dốc = ms/ảnh. Paper: CPU {PAPER_CPU_SLOPE}, GPU {PAPER_GPU_SLOPE}.")
    save(fig, figdir, "fig1_conv_time")


def fig_speedup(df, impl, baseline, meta, figdir, name, title, xlabel, log_x, note, runs=None):
    table, spread = speedup_table(df[df["impl"] == impl], baseline), None
    if runs is not None and table is not None:
        # Speedup inside each run (same machine state), then mean and std over the runs.
        per_run = pd.concat({r: speedup_table(d[d["impl"] == impl], baseline)
                             for r, d in runs.groupby("run")})
        table, spread = per_run.groupby(level=1).mean(), per_run.groupby(level=1).std()
    if table is None or table.empty:
        print(f"skip {name}: need the baseline and at least one other device/thread config")
        return None
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    labels = []
    for key, (label, color, _) in meta.items():
        if key not in table:
            continue
        s = table[key].dropna()
        err = None if spread is None else spread[key].reindex(s.index).to_numpy(float)
        scatter_line(ax, s.index.to_numpy(float), s.to_numpy(float), color, label, err=err)
        labels.append((s.index[-1], s.iloc[-1], f"{label}\n{s.iloc[-1]:.1f}× ở {int(s.index[-1])}", color))
    ax.axhline(1, color=AXIS, lw=0.8, zorder=1)
    if log_x:
        int_ticks(ax, pow2(table.index), log=True)
    else:
        ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"Speedup so với {meta[baseline][0]} (lần)")
    ax.set_title(title, pad=30)
    top_legend(ax)
    end_labels(ax, labels, min_gap=0.12)
    footnote(fig, note)
    save(fig, figdir, name)
    return table


def fig_fc_time(fc, op, meta, figdir, name, title, summary, expected, min_n):
    df = fc[fc["impl"] == op]
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    labels = []
    for key, (label, color, _) in meta.items():
        d = df[df["config"] == key].sort_values("size")
        if d.empty:
            continue
        n, t = d["size"].to_numpy(float), d["mean_ms"].to_numpy(float)
        k = powfit(n, t, min_n)
        summary.append((op, key, "exponent", k))
        summary.append((op, key, "exponent_n1024", powfit(n, t, 1024)))
        summary.append((op, key, "time_ms_at_max_n", t[-1]))
        summary.append((op, key, "gflops_at_max_n", float(d["gflops"].iloc[-1])))
        summary.append((op, key, "gbps_at_max_n", float(d["gbps"].iloc[-1])))
        scatter_line(ax, n, t, color, label, err=yerr(d))
        labels.append((n[-1], t[-1], f"{label}\nt $\\propto$ n$^{{{k:.2f}}}$", color))
    log_y(ax)
    int_ticks(ax, pow2(df["size"].unique()), log=True)
    ax.set_xlabel("Kích thước n (ma trận n×n)")
    ax.set_ylabel("Thời gian (ms, thang log)")
    ax.set_title(title, pad=30)
    top_legend(ax)
    end_labels(ax, labels, min_gap=0.12)
    footnote(fig, f"Trục log–log: độ dốc của đường là số mũ k trong t ~ n^k (lý thuyết: k = {expected}), "
                  f"khớp trên các n ≥ {min_n}. {reps(df).capitalize()}.")
    save(fig, figdir, name)


def fig_conv_impls(conv, meta, figdir):
    keys = [k for k in meta if k in set(conv["config"])]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 4.0), squeeze=False)
    for ax, key in zip(axes[0], keys):
        d = conv[conv["config"] == key]
        for i, impl in enumerate(IMPL_ORDER):
            s = d[d["impl"] == impl].sort_values("size")
            if s.empty:
                continue
            per_img = s["mean_ms"].to_numpy(float) / s["size"].to_numpy(float)
            scatter_line(ax, s["size"].to_numpy(float), per_img, SLOTS[i], IMPL_LABEL[impl])
        log_y(ax)
        ax.set_xlim(left=0)
        ax.set_title(meta[key][0], fontsize=11)
        ax.set_xlabel("Batch size")
    axes[0][0].set_ylabel("Thời gian mỗi ảnh (ms, thang log)")
    fig.subplots_adjust(top=0.74, wspace=0.28)
    fig.suptitle("Hình 5 · So sánh ba cách cài đặt lớp convolution (mở rộng)", x=0.06, y=0.99,
                 ha="left", fontsize=12, fontweight="bold", color=INK)
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.06, 0.93), ncol=3)
    footnote(fig, "Mỗi khung một thiết bị, trục y riêng cho từng khung. Đường càng thấp càng nhanh; "
                  "đường giảm dần theo batch size nghĩa là gộp batch giúp tận dụng phần cứng tốt hơn. "
                  "Trên CPU, thời gian của thư viện có kèm việc cấp phát bộ nhớ đầu ra ở mỗi lần gọi.",
             width=60 * len(keys))
    save(fig, figdir, "fig5_conv_impls")


# ---------------------------------------------------------------- summary

def write_summary(run_dir, figdir, env, meta, summary, conv_speed, fc_speed, baseline):
    rows = pd.DataFrame(summary, columns=["metric_group", "config", "metric", "value"])
    rows["label"] = rows["config"].map(lambda k: meta.get(k, (k,))[0])
    rows.to_csv(figdir / "summary.csv", index=False)

    def get(group, key, metric):
        hit = rows[(rows.metric_group == group) & (rows.config == key) & (rows.metric == metric)]
        return float(hit["value"].iloc[0]) if not hit.empty else None

    lines = [f"# Tóm tắt kết quả — {env.get('machine', run_dir.name)} / {run_dir.name}", "",
             f"- CPU: {env.get('cpu', '?')} (torch dùng tối đa {env.get('torch_default_threads', '?')} luồng)",
             f"- GPU: {env.get('gpu', 'không có')}"
             + (f" · CUDA {env.get('cuda')} · cuDNN {env.get('cudnn')}" if env.get("cuda_available") else ""),
             f"- PyTorch {env.get('torch', '?')} · Python {env.get('python', '?')} · {env.get('os', '')}", "",
             "## Thí nghiệm 1 — lớp convolution, cài đặt kiểu Caffe (so với Hình 1–2 của paper)", "",
             "| Cấu hình | Độ dốc (ms/ảnh) | R² | Tỷ lệ độ dốc so với baseline |",
             "|---|---|---|---|",
             f"| *Paper: Xeon E5-2670 (CPU)* | *{PAPER_CPU_SLOPE}* | | *1×* |",
             f"| *Paper: GRID K520 (GPU)* | *{PAPER_GPU_SLOPE}* | | *{PAPER_CPU_SLOPE / PAPER_GPU_SLOPE:.1f}×* |"]
    base_slope = get("conv_caffe", baseline, "slope_ms_per_image")
    for key, (label, _, _) in meta.items():
        slope = get("conv_caffe", key, "slope_ms_per_image")
        if slope is None:
            continue
        ratio = f"{base_slope / slope:.1f}×" if base_slope else ""
        lines.append(f"| {label} | {slope:.4g} | {get('conv_caffe', key, 'r2'):.4f} | {ratio} |")
    if conv_speed is not None:
        lines += ["", "Speedup theo batch size (thời gian baseline / thời gian cấu hình):", "",
                  "| Batch | " + " | ".join(meta[k][0] for k in conv_speed.columns) + " |",
                  "|---" * (len(conv_speed.columns) + 1) + "|"]
        for size, row in conv_speed.iterrows():
            lines.append(f"| {size} | " + " | ".join(f"{v:.2f}×" for v in row) + " |")

    lines += ["", "## Thí nghiệm 2 — lớp fully-connected (so với Hình 3–4 của paper)", "",
              "| Cấu hình | Phép | Số mũ k (t ∝ nᵏ) | k nếu khớp từ n ≥ 1024 | Thời gian ở n lớn nhất (ms) "
              "| GFLOP/s | GB/s |",
              "|---|---|---|---|---|---|---|"]
    for op, expected in (("matvec", 2), ("gemm", 3)):
        for key, (label, _, _) in meta.items():
            k = get(op, key, "exponent")
            if k is None:
                continue
            lines.append(f"| {label} | {op} (lý thuyết k={expected}) | {k:.2f} | "
                         f"{get(op, key, 'exponent_n1024'):.2f} | "
                         f"{get(op, key, 'time_ms_at_max_n'):.3f} | {get(op, key, 'gflops_at_max_n'):.1f} | "
                         f"{get(op, key, 'gbps_at_max_n'):.1f} |")
    if fc_speed is not None:
        lines += ["", "Speedup matvec theo n:", "",
                  "| n | " + " | ".join(meta[k][0] for k in fc_speed.columns) + " |",
                  "|---" * (len(fc_speed.columns) + 1) + "|"]
        for size, row in fc_speed.iterrows():
            lines.append(f"| {size} | " + " | ".join(f"{v:.2f}×" for v in row) + " |")
    lines += ["", "Ghi chú: trục Hình 1 của paper là Time (ms) theo batch size, nên độ dốc là ms/ảnh. "
              "Con số tuyệt đối khác paper vì phần cứng khác đời; điều cần so là xu hướng."]
    path = figdir / "summary.md"
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("saved", path)


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("run_dir", type=Path)
    parser.add_argument("--baseline-threads", type=int, default=1,
                        help="CPU thread count used as the baseline (paper: single core)")
    parser.add_argument("--figdir", type=Path, default=None)
    # Matvec is memory-bound: below ~4096 the matrix (<= 16 MiB at n=2048) still fits in
    # the last-level cache, so the fit starts where the data streams from DRAM.
    parser.add_argument("--fit-min-n-matvec", type=int, default=4096)
    parser.add_argument("--fit-min-n-gemm", type=int, default=1024)
    args = parser.parse_args()

    run_dir = args.run_dir.resolve()
    figdir = (args.figdir or run_dir / "figures").resolve()
    figdir.mkdir(parents=True, exist_ok=True)
    env, frames = load_run(run_dir)
    if not frames:
        raise SystemExit(f"no conv.csv / fc.csv in {run_dir}")
    apply_style()

    keys = set().union(*(set(df["config"]) for df in frames.values()))
    baseline = f"cpu-{args.baseline_threads}"
    if baseline not in keys:
        cpu = sorted((k for k in keys if k.startswith("cpu-")), key=lambda k: int(k[4:]))
        baseline = cpu[0] if cpu else sorted(keys)[0]
    meta = series_meta(keys, env, baseline)

    summary, conv_speed, fc_speed = [], None, None
    if "conv" in frames:
        conv = frames["conv"]
        fig_conv_time(conv, meta, figdir, summary)
        conv_speed = fig_speedup(conv, "caffe", baseline, meta, figdir, "fig2_conv_speedup",
                                 "Hình 2 · Lớp convolution: speedup theo batch size",
                                 "Batch size (số ảnh)", False,
                                 "Speedup = thời gian của baseline / thời gian của cấu hình, cùng batch size, "
                                 "cài đặt kiểu Caffe. Đường ngang: 1× (bằng baseline).",
                                 frames.get("runs_conv"))
        fig_conv_impls(conv, meta, figdir)
    if "fc" in frames:
        fc = frames["fc"]
        fig_fc_time(fc, "matvec", meta, figdir, "fig3_fc_time",
                    "Hình 3 · Lớp fully-connected (nhân ma trận–vector): thời gian theo n", summary, 2,
                    args.fit_min_n_matvec)
        fc_speed = fig_speedup(fc, "matvec", baseline, meta, figdir, "fig4_fc_speedup",
                               "Hình 4 · Nhân ma trận–vector: speedup theo n",
                               "Kích thước n (ma trận n×n)", True,
                               "Speedup = thời gian của baseline / thời gian của cấu hình, cùng n. "
                               "Đường ngang: 1× (bằng baseline).", frames.get("runs_fc"))
        fig_fc_time(fc, "gemm", meta, figdir, "fig6_gemm_time",
                    "Hình 6 · Nhân ma trận–ma trận: thời gian theo n (mở rộng)", summary, 3,
                    args.fit_min_n_gemm)
    write_summary(run_dir, figdir, env, meta, summary, conv_speed, fc_speed, baseline)


if __name__ == "__main__":
    main()

## 3 · Helper: run a command, stream its output, stop the notebook if it fails

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
WORK = Path("/kaggle/working")
CODE, RES = WORK / "code", WORK / "results" / "kaggle"
# glibc: serve big buffers from the heap and keep them, so F.conv2d on the CPU does not pay
# page faults for a fresh output buffer on every call (recorded in env.json).
ENV = dict(os.environ, MALLOC_MMAP_MAX_="0", MALLOC_TRIM_THRESHOLD_="68719476736", PYTHONUNBUFFERED="1")

def py(*args):
    cmd = [sys.executable, *map(str, args)]
    p = subprocess.Popen(cmd, cwd=CODE, env=ENV, text=True, encoding="utf-8",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"exit code {p.returncode}: {' '.join(cmd)}")

## 4 · Smoke test (a few seconds, kept out of the results)

In [ ]:
shutil.rmtree(WORK / "smoke", ignore_errors=True)
py("run_all.py", "--machine", "kaggle", "--quick", "--tag", "smoke", "--outdir", WORK / "smoke",
   "--devices", "cuda,cpu", "--warm-seconds", 1, "--order-seed", 1)

## 5 · Diagnostic: cold GPU, ascending batch sizes (the conditions of the first Kaggle run)

Reproduces the jump seen around batch 96-128, now with the SM clock and the host issue time.

In [ ]:
import time
from datetime import datetime
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
time.sleep(30)                                  # let the GPU fall back to its idle clocks
DIAG = RES / f"diag_cold_{STAMP}"
py("run_all.py", "--machine", "kaggle", "--tag", DIAG.name, "--outdir", RES.parent,
   "--devices", "cuda", "--skip", "fc", "--warm-seconds", 0, "--order-seed", 0)

## 6 · 3 full runs, GPU first (warm GPU, shuffled sizes, 30 repeats per point)

In [ ]:
RUN1 = RES / f"run1_{STAMP}"
py("run_all.py", "--machine", "kaggle", "--tag", RUN1.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 30, "--order-seed", 1)

In [ ]:
RUN2 = RES / f"run2_{STAMP}"
py("run_all.py", "--machine", "kaggle", "--tag", RUN2.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 30, "--order-seed", 2)

In [ ]:
RUN3 = RES / f"run3_{STAMP}"
py("run_all.py", "--machine", "kaggle", "--tag", RUN3.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 30, "--order-seed", 3)

## 7 · Combine the runs, figures, summary and GPU diagnosis

In [ ]:
AGG = RES / f"agg_{STAMP}"
py("aggregate.py", RUN1, RUN2, RUN3, "--out", AGG)
py("plot.py", AGG)
py("gpu_diag.py", DIAG, RUN1, RUN2, RUN3, "--out", AGG / "gpu_diag.md")
from IPython.display import Image, Markdown, display
display(Markdown((AGG / "figures" / "summary.md").read_text(encoding="utf-8")))
for png in sorted((AGG / "figures").glob("*.png")):
    display(Image(filename=str(png), width=900))

## 8 · Package the results for download

In [ ]:
print(shutil.make_archive(str(WORK / "results_kaggle"), "zip", WORK, "results"))